In [1]:
# ================================================================
# 항공 채용공고 텍스트 분석 — Gradio 웹 앱 (코드 셀 1개)
# ================================================================
# [전체 흐름]
# ① 필수 패키지 준비 → ② Gradio 웹 화면 실행 → ③ 웹에서 CSV 파일 선택
# ④ 파일 인코딩 자동 확인 → ⑤ 제목 정제·중복 제거·형태소 분석
# ⑥ 직무·고용형태·지역 분류 → ⑦ 키워드·2-gram·TF-IDF·연도별 분석
# ⑧ 웹 화면에 표·차트 표시 → ⑨ 공고 검색 → ⑩ 정제 CSV 다운로드
#
# 실행 방법: Google Colab의 이 셀을 Shift+Enter로 딱 한 번 실행하세요.
#          출력된 Gradio 링크에서 CSV를 선택하고 '전체 분석 시작'을 누르세요.
# 입력 파일: '채용정보'(공고 제목), '회사', '시작일자', '주소' 열이 들어 있는 CSV
# 안내: 공고 제목에 적힌 표현만 분석합니다. 공고 수는 실제 채용 인원이 아닙니다.

# ── 0. 라이브러리 설치: 현재 환경에 없는 패키지만 설치합니다. ──
import sys
import subprocess
import importlib.util
for module, package in [("gradio", "gradio>=4.44,<7"), ("kiwipiepy", "kiwipiepy"),
                        ("sklearn", "scikit-learn"), ("matplotlib", "matplotlib"),
                        ("pandas", "pandas"), ("koreanize_matplotlib", "koreanize-matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# ── 1. 분석에 필요한 도구를 불러옵니다. ──
import re                               # 정규식: 제목 안에서 특정 단어 패턴을 찾거나 지움
import tempfile                         # 결과 CSV를 안전한 임시 폴더에 저장
from pathlib import Path               # 파일 경로를 다룸
from collections import Counter        # 키워드·두 단어 묶음 등장 횟수를 셈
import gradio as gr                    # 브라우저에 업로드·검색·결과 화면을 만듦
import pandas as pd                    # CSV 데이터와 표(DataFrame)를 다룸
import matplotlib.pyplot as plt         # 결과 그래프를 그림
import koreanize_matplotlib            # Matplotlib의 한글 글꼴을 설정
from kiwipiepy import Kiwi             # 한국어 형태소 분석기
from sklearn.feature_extraction.text import TfidfVectorizer  # 대표 키워드의 TF-IDF 점수

kiwi = Kiwi()                          # 제목에서 명사와 영어 단어를 추출할 분석기 준비

# ── 2. 분류 규칙: 학습자가 가장 쉽게 수정할 수 있는 부분 ──
# 딕셔너리의 왼쪽은 결과 이름, 오른쪽은 제목에서 찾을 정규식입니다.
# 정규식의 | 는 '또는'을 뜻합니다. 예: '계약|기간제'는 둘 중 하나만 있어도 일치
EMPLOY = {
    "신입": r"신입", "경력": r"경력", "인턴": r"인턴", "정규직": r"정규",
    "계약직": r"계약|기간제|임기제", "아르바이트": r"아르바이트|알바|시간제|단기"
}
JOBS = {
    "지상조업·여객서비스": r"지상|조업|수하물|램프|탑재|발권|예약|카운터|라운지|여객|운송|화물|상담",
    "드론·UAM": r"드론|무인|UAM|도심항공",
    "연구·개발": r"연구|개발|설계|해석|소프트웨어|하드웨어|엔지니어",
    "정비": r"정비|MRO|기체|엔진",
    "사무·경영": r"사무|행정|경영|회계|인사|총무|마케팅|구매|영업|재무|기획",
    "조종·운항": r"조종|기장|운항|비행|파일럿",
    "교육": r"교관|강사|교원|교수|교육",
    "보안·검색": r"보안|검색|경비",
    "식음·기내식": r"기내식|조리|식당|케이터링",
    "객실승무원": r"승무원|객실",
}
STOP = {"채용", "모집", "공고", "공개", "직원", "사원", "인력", "근로자", "항공",
        "직", "팀", "부", "원", "급", "사", "분야", "부문", "본부", "업무", "담당", "담당자",
        "상반기", "하반기", "수시", "상시", "신규", "기간", "경쟁", "위촉",
        "신입", "경력", "인턴", "정규직", "계약직", "계약"}
NORM = {"인천국제공항": "인천공항", "제주국제공항": "제주공항",
        "김포국제공항": "김포공항", "김해국제공항": "김해공항"}
KOREA = ("서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종", "경기", "강원",
         "충청북", "충청남", "전북", "전라북", "전라남", "경상북", "경상남", "제주")

# ── 3. CSV 읽기: 공공데이터에서 흔한 여러 한글 인코딩을 차례대로 시험합니다. ──
def read_csv(path):
    # 파일 객체가 아니라 gr.File에서 받은 실제 임시 파일 경로입니다.
    for encoding in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            data = pd.read_csv(path, encoding=encoding, dtype=str, low_memory=False)
            # 올바른 CSV에는 '채용정보'라는 열이 있어야 합니다.
            data.columns = data.columns.astype(str).str.strip()
            if "채용정보" in data.columns:
                return data.fillna(""), encoding
        except (UnicodeError, UnicodeDecodeError):
            continue
    raise ValueError("CSV를 읽지 못했습니다. UTF-8 또는 CP949 CSV이고 '채용정보' 열이 있는지 확인하세요.")

# ── 4. 제목 정제: 공고마다 달라지는 장식·차수·인원·기호를 없앱니다. ──
def clean(title):
    title = re.sub(r"\[[^\]]*\]", " ", title)      # [경력] 등 대괄호 태그 제거
    title = re.sub(r"[’'‘]?\d{2,4}\s*년|\d+\s*(차|기|명|월|일)", " ", title)
    title = re.sub(r"[^가-힣A-Za-z]", " ", title)  # 한글·영어만 남김
    return re.sub(r"\s+", " ", title).strip()      # 중복 공백 제거

# ── 5. 형태소 분석: 명사(NNG, NNP)와 외국어(SL)만 남깁니다. ──
def to_keywords(title):
    tokens = kiwi.tokenize(title)                   # 문장을 형태소 단위로 나눔
    words = [t.form.upper() if t.tag == "SL" else t.form
             for t in tokens if t.tag in ("NNG", "NNP", "SL")]
    words = [NORM.get(w, w) for w in words]        # 같은 장소의 다른 표기를 통일
    return [w for w in words if len(w) >= 2 and w not in STOP]

# ── 6. 직무·고용형태는 한 제목에서 복수로 탐지할 수 있습니다. ──
def find_jobs(title):
    found = [name for name, pattern in JOBS.items() if re.search(pattern, title, re.I)]
    return found if found else ["미분류"]

def find_employ(title):
    return [name for name, pattern in EMPLOY.items() if re.search(pattern, title, re.I)]

# ── 7. 공통 차트 함수: Series(이름→개수)를 가로 막대로 표시합니다. ──
def bar_chart(values, title, color="#2563eb"):
    fig, ax = plt.subplots(figsize=(10, 5))
    if len(values):
        values = values.iloc[::-1]                  # 가장 큰 막대를 위에 표시
        ax.barh(values.index.astype(str), values.values, color=color)
        ax.set_xlim(0, max(values.values) * 1.15)
        ax.grid(axis="x", alpha=0.18)
        ax.set_axisbelow(True)
    else:
        ax.text(0.5, 0.5, "표시할 결과가 없습니다", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title)
    fig.tight_layout()
    return fig

# ── 8. TF-IDF: 직무별 또는 연도별로 특징적인 단어를 구합니다. ──
def tfidf_keywords(data, group_col, max_df=1.0):
    # 같은 집단의 제목 키워드를 하나의 '문서'로 이어 붙입니다.
    # 예: '정비' 직무에 속한 공고 키워드를 하나의 긴 문서로 연결
    grouped = data.groupby(group_col)["키워드문자열"].agg(" ".join)
    grouped = grouped[grouped.str.strip() != ""]  # 빈 문서는 제외
    if len(grouped) < 2:                           # 비교 문서가 1개뿐이면 TF-IDF 비교가 어려움
        return pd.DataFrame(columns=[group_col, "대표 키워드"])
    vectorizer = TfidfVectorizer(analyzer=str.split, sublinear_tf=True, max_df=max_df)
    try:
        matrix = vectorizer.fit_transform(grouped)
    except ValueError:                             # 단어가 전부 제거되어 어휘가 없을 때
        return pd.DataFrame(columns=[group_col, "대표 키워드"])
    names = vectorizer.get_feature_names_out()
    result = []
    for i, label in enumerate(grouped.index):
        scores = matrix.getrow(i).toarray().ravel()
        top = [names[j] for j in scores.argsort()[::-1][:7] if scores[j] > 0]
        result.append([label, ", ".join(top)])
    return pd.DataFrame(result, columns=[group_col, "대표 키워드"])

# ── 9. 버튼을 누르면 실행할 핵심 분석 함수 ──
def analyze(file_path):
    if not file_path:
        raise gr.Error("먼저 CSV 파일을 선택하세요.")
    try:
        raw, encoding = read_csv(file_path)
        if raw.empty:
            raise ValueError("CSV에 공고 데이터가 없습니다.")
        # 일부 CSV에는 회사·날짜·주소 열이 없을 수 있어 빈 열로 보완합니다.
        for col in ("회사", "시작일자", "주소"):
            if col not in raw.columns:
                raw[col] = ""
        raw["채용정보"] = raw["채용정보"].astype(str).str.strip()
        raw = raw[raw["채용정보"] != ""].copy()  # 제목이 없는 행은 분석하지 않음
        if raw.empty:
            raise ValueError("채용정보 열에 분석할 제목이 없습니다.")
        raw["연도"] = raw["시작일자"].str.extract(r"(\d{4})", expand=False).fillna("")
        raw["태그"] = raw["채용정보"].str.findall(r"\[([^\]]+)\]").str.join(", ")
        raw["고용형태목록"] = raw["채용정보"].apply(find_employ)
        first_address = raw["주소"].str.split().str[0].fillna("")
        raw["지역"] = first_address.apply(
            lambda item: item if item.startswith(KOREA) else "해외·기타")
        raw["제목"] = raw["채용정보"].apply(clean)
        raw = raw[raw["제목"] != ""].copy()
        # 반복 공고는 '회사+정제 제목'이 모두 같은 경우로 간주합니다.
        # 날짜가 달라도 합쳐지므로 '공고 건수 추정'이 아닌 '중복 제거된 제목 수'로 이해합니다.
        df = raw.drop_duplicates(["회사", "제목"]).reset_index(drop=True)
        if df.empty:
            raise ValueError("제목 정제 후 남은 공고가 없습니다.")
        df["직무목록"] = df["제목"].apply(find_jobs)
        df["키워드목록"] = df["제목"].apply(to_keywords)
        df["키워드문자열"] = df["키워드목록"].str.join(" ")
        df["직무"] = df["직무목록"].str.join(", ")
        df["고용형태"] = df["고용형태목록"].str.join(", ")
        df["키워드"] = df["키워드목록"].str.join(", ")

        # 빈도 분석: 전체 제목에서 각 키워드가 등장한 횟수
        freq = Counter(word for words in df["키워드목록"] for word in words)
        top20 = pd.Series(dict(freq.most_common(20)), dtype="int64")
        # 2-gram: 각 공고 안에서 서로 이웃한 키워드 두 개를 셉니다.
        bigrams = Counter(" ".join(words[i:i+2]) for words in df["키워드목록"]
                          for i in range(len(words)-1))
        top15 = pd.Series(dict(bigrams.most_common(15)), dtype="int64")
        job_counts = df["직무목록"].explode().value_counts()
        # 한 공고에 직무가 2개 이상이면 여러 직무에 각각 포함됩니다.
        job_table = job_counts.rename_axis("직무").reset_index(name="공고 수")
        keyword_table = top20.rename_axis("키워드").reset_index(name="등장 횟수")
        bigram_table = top15.rename_axis("2-gram").reset_index(name="등장 횟수")
        job_long = df.explode("직무목록")
        job_tfidf = tfidf_keywords(job_long.rename(columns={"직무목록":"분류"}), "분류")
        job_tfidf = job_tfidf.rename(columns={"분류":"직무"})

        # 연도별 추이: 각 연도의 공고 수를 분모로 하여 직무 비율(%)을 구함
        valid_year = df[df["연도"].str.fullmatch(r"\d{4}") & (df["연도"] >= "2019")]
        if len(valid_year):
            yr_long = valid_year.explode("직무목록")
            yr_counts = pd.crosstab(yr_long["연도"], yr_long["직무목록"])
            yr_ratio = yr_counts.div(valid_year["연도"].value_counts(), axis=0).mul(100)
            yr_ratio = yr_ratio.fillna(0).sort_index()
            year_table = yr_ratio.round(1).reset_index()
            year_tfidf = tfidf_keywords(valid_year, "연도", max_df=0.6)
        else:
            yr_ratio = pd.DataFrame()
            year_table = pd.DataFrame(columns=["연도"])
            year_tfidf = pd.DataFrame(columns=["연도", "대표 키워드"])

        # 지역별 분석: 주요 지역에서 직무 키워드가 얼마나 나타나는지 계산
        regions = df["지역"].value_counts().head(6).index
        rt_counts = pd.crosstab(job_long["지역"], job_long["직무목록"])
        region_ratio = rt_counts.reindex(index=regions, columns=list(JOBS), fill_value=0)
        region_ratio = region_ratio.div(df["지역"].value_counts().reindex(regions), axis=0).mul(100)
        region_table = region_ratio.round(1).reset_index().rename(columns={"지역":"지역"})

        # 고용형태: 제목에 '신입·경력' 등 문구가 있는 경우만 세는 탐지 횟수
        employ_counts = df["고용형태목록"].explode().dropna().value_counts()
        employ_table = employ_counts.rename_axis("고용형태").reset_index(name="공고 수")
        # 기업별 공고 건수 TOP 10: 중복 제거된 제목 기준
        company_table = df[df["회사"].str.strip() != ""]["회사"].value_counts().head(10).rename_axis("회사").reset_index(name="공고 수")

        # 첫 화면에 보이는 요약 메시지
        unclassified = (df["직무"] == "미분류").mean() * 100
        summary = (f"### 분석 완료\n"
                   f"- 파일 인코딩: **{encoding}** / 입력 공고: **{len(raw):,}건** / "
                   f"중복 제거 후: **{len(df):,}건**\n"
                   f"- 중복·반복 제거: **{len(raw)-len(df):,}건** / 미분류: **{unclassified:.1f}%**\n"
                   f"- 직무는 복수 선택 가능하며, 수치는 **실제 채용 인원**이 아닙니다.")

        # 연도별 꺾은선 그래프: 연도·직무가 존재하는 경우에만 표시
        fig_year, ax = plt.subplots(figsize=(10, 5))
        cols = [c for c in ["지상조업·여객서비스", "드론·UAM", "연구·개발", "정비", "조종·운항"] if c in yr_ratio.columns]
        if cols:
            for col in cols:
                ax.plot(yr_ratio.index, yr_ratio[col], marker="o", label=col)
            ax.legend(loc="best", fontsize=9)
            ax.set_ylabel("그 해 공고 중 비율 (%)")
            ax.grid(alpha=0.2)
        else:
            ax.text(0.5, 0.5, "연도별 분석에 필요한 데이터가 없습니다", ha="center", transform=ax.transAxes)
        ax.set_title("연도별 직무 비율 변화 (2019년 이후)")
        fig_year.tight_layout()

        # 지역×직무 그래프: 값이 없는 셀도 0%로 표시
        fig_region, axr = plt.subplots(figsize=(13, 5))
        if len(region_ratio):
            img = axr.imshow(region_ratio.values, cmap="Blues", aspect="auto")
            axr.set_xticks(range(len(region_ratio.columns)), region_ratio.columns, rotation=35, ha="right")
            axr.set_yticks(range(len(region_ratio.index)), region_ratio.index)
            fig_region.colorbar(img, ax=axr, label="공고 비율 (%)")
        else:
            axr.text(0.5, 0.5, "지역 분석 결과가 없습니다", ha="center", transform=axr.transAxes)
        axr.set_title("지역별 직무 비율 (%)")
        fig_region.tight_layout()

        # 정제 결과 CSV: 엑셀에서 한글이 깨지지 않도록 utf-8-sig로 저장
        export_cols = ["연도", "시작일자", "회사", "지역", "채용정보", "제목", "태그", "고용형태", "키워드", "직무"]
        saved = Path(tempfile.mkdtemp(prefix="air_jobs_")) / "job_dataset.csv"
        df[export_cols].to_csv(saved, index=False, encoding="utf-8-sig")
        preview = df[export_cols].head(30)
        # 검색에서 재사용할 데이터는 gr.State에 보관합니다.
        return (summary, preview, keyword_table, bar_chart(top20,"키워드 TOP 20"),
                bigram_table, bar_chart(top15,"2-gram TOP 15"),
                job_table, bar_chart(job_counts.head(12),"직무별 공고 수"), job_tfidf,
                year_table, fig_year, year_tfidf, region_table, fig_region,
                employ_table, bar_chart(employ_counts,"고용형태 표현별 공고 수"),
                company_table, str(saved), df[export_cols], "파일 분석 완료! 검색어를 입력하고 '공고 검색'을 누르세요.")
    except Exception as e:
        raise gr.Error(f"분석 실패: {e}") from e

# ── 10. 검색: 이미 분석한 공고를 제목·회사·지역·직무로 검색합니다. ──
def search_jobs(query, job, data):
    if data is None:
        return pd.DataFrame(columns=["연도", "회사", "채용정보", "지역", "직무", "고용형태"]), "먼저 CSV를 분석하세요."
    selected = data.copy()
    # re.escape(): 검색어에 괄호·점 등이 있어도 정규식으로 해석하지 않고 문자 그대로 찾음
    if query.strip():
        pattern = re.escape(query.strip())
        mask = (selected["채용정보"].str.contains(pattern, case=False, na=False) |
                selected["회사"].str.contains(pattern, case=False, na=False) |
                selected["지역"].str.contains(pattern, case=False, na=False) |
                selected["키워드"].str.contains(pattern, case=False, na=False))
        selected = selected[mask]
    if job != "전체":
        selected = selected[selected["직무"].str.contains(re.escape(job), regex=True, na=False)]
    return selected.head(100).reset_index(drop=True), f"검색 결과: **{len(selected):,}건** (표에는 최대 100건 표시)"

# ── 11. Gradio 웹 화면: 파일 선택과 결과 확인을 모두 브라우저에서 합니다. ──
with gr.Blocks(title="항공 채용공고 NLP 분석") as demo:
    gr.Markdown("# ✈️ 항공 채용공고 텍스트 분석")
    gr.Markdown("**실습 흐름:** CSV 업로드 → 제목 전처리 → Kiwi 키워드 추출 → 직무 분류 → 통계·TF-IDF → 공고 검색 → 결과 다운로드")
    with gr.Row():
        csv_file = gr.File(label="① 분석할 채용공고 CSV 선택", file_types=[".csv"], type="filepath")
        start_btn = gr.Button("② 전체 분석 시작", variant="primary")
    status = gr.Markdown("CSV를 선택한 다음 **전체 분석 시작**을 누르세요.")
    saved_data = gr.State(value=None)    # 파일 업로드 결과 데이터는 브라우저 세션 단위로 저장

    with gr.Tab("1. 정제 결과"):
        gr.Markdown("태그·고용형태·지역을 추출하고 제목을 정제한 뒤 중복 공고를 제거합니다.")
        preview = gr.Dataframe(label="정제 데이터 미리보기 (30건)", interactive=False)
        download = gr.File(label="정제 CSV 다운로드 (job_dataset.csv)")
    with gr.Tab("2. 키워드 · 2-gram"):
        with gr.Row():
            keyword_plot = gr.Plot(label="키워드 빈도")
            keyword_df = gr.Dataframe(label="키워드 TOP 20", interactive=False)
        with gr.Row():
            bigram_plot = gr.Plot(label="두 단어 묶음")
            bigram_df = gr.Dataframe(label="2-gram TOP 15", interactive=False)
    with gr.Tab("3. 직무 분류 · TF-IDF"):
        with gr.Row():
            job_plot = gr.Plot(label="직무별 공고 수")
            job_df = gr.Dataframe(label="직무별 공고 수", interactive=False)
        job_tfidf_df = gr.Dataframe(label="직무별 대표 키워드 (TF-IDF)", interactive=False)
    with gr.Tab("4. 연도별 변화"):
        year_plot = gr.Plot(label="연도별 직무 비율")
        year_df = gr.Dataframe(label="연도별 직무 비율 (%)", interactive=False)
        year_tfidf_df = gr.Dataframe(label="연도별 특징 키워드 (TF-IDF)", interactive=False)
    with gr.Tab("5. 지역 · 고용형태 · 회사"):
        region_plot = gr.Plot(label="지역 × 직무")
        region_df = gr.Dataframe(label="지역별 직무 비율 (%)", interactive=False)
        employ_plot = gr.Plot(label="고용형태")
        employ_df = gr.Dataframe(label="고용형태 분포", interactive=False)
        company_df = gr.Dataframe(label="공고가 많은 회사 TOP 10", interactive=False)
    with gr.Tab("6. 공고 검색"):
        gr.Markdown("원본 제목·회사·지역·키워드에서 검색하고, 필요하면 직무를 함께 선택합니다.")
        with gr.Row():
            search_text = gr.Textbox(label="검색어", placeholder="예: 정비, 인천, 드론, 항공사")
            job_choice = gr.Dropdown(choices=["전체"] + list(JOBS) + ["미분류"], value="전체", label="직무")
            search_btn = gr.Button("공고 검색", variant="primary")
        search_status = gr.Markdown("파일 분석 후 검색할 수 있습니다.")
        search_result = gr.Dataframe(label="검색된 공고 (최대 100건)", interactive=False)
    gr.Markdown("**해석 주의:** 여러 직무·고용형태가 하나의 제목에 함께 등장할 수 있습니다. 제목만으로 구한 분류이며 시장 전체의 채용 추세를 확정할 수 없습니다.")

    # outputs의 순서와 analyze()의 return 값 순서가 정확히 같아야 화면이 연결됩니다.
    start_btn.click(
        fn=analyze, inputs=csv_file,
        outputs=[status, preview, keyword_df, keyword_plot, bigram_df, bigram_plot,
                 job_df, job_plot, job_tfidf_df, year_df, year_plot, year_tfidf_df,
                 region_df, region_plot, employ_df, employ_plot, company_df,
                 download, saved_data, search_status])
    search_btn.click(fn=search_jobs, inputs=[search_text, job_choice, saved_data],
                     outputs=[search_result, search_status])
    search_text.submit(fn=search_jobs, inputs=[search_text, job_choice, saved_data],
                       outputs=[search_result, search_status])

# Colab에서도 외부 브라우저로 접속할 수 있도록 링크를 표시합니다.
# share=True로 생성되는 공개 링크에는 개인·민감한 채용 데이터는 올리지 마세요.
import os
is_colab = "google.colab" in sys.modules
# 코랩: 외부 접속 가능한 공유 링크, 로컬 Jupyter: 로컬 브라우저 주소
try:
    demo.launch(share=is_colab, debug=False, show_error=True)
except Exception as exc:
    print("웹 화면 실행 중 오류:", exc)
    print("환경에 따라 런타임을 다시 시작한 후 셀을 재실행해야 할 수 있습니다.")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6f124fb9e147ba3a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
